In [65]:
from cfb.config import ROOT, SQL_ANALYSIS_DIR
from cfb.db import engine, read_query
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# modeling (scikit-learn)
import joblib                                                   # save / load the trained model
from sklearn.linear_model import LogisticRegression             # baseline model
from sklearn.ensemble import HistGradientBoostingClassifier     # main model: handles NULLs, monotonic constraints
from sklearn.pipeline import (make_pipeline, Pipeline)
from sklearn.impute import SimpleImputer                        # fill NULLs for the baseline (logistic can't take them)
from sklearn.preprocessing import (StandardScaler,SplineTransformer)
from sklearn.isotonic import IsotonicRegression                 # calibrate probabilities on the validation season
from sklearn.calibration import calibration_curve, CalibrationDisplay
from sklearn.metrics import log_loss, brier_score_loss, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer

from sklearn.metrics import (
    classification_report,
    ConfusionMatrixDisplay,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_curve,
    roc_auc_score,
)

In [66]:
fg = pd.read_sql(
    "SELECT * FROM marts.fourth_down_features WHERE decision = 'field goal' AND season >= 2008",
    engine,
)
fg

,play_id,game_id,drive_id,season,week,offense,defense,decision,distance,yards_to_goal,...,off_success_szn,def_ppa_szn,off_elo,def_elo,elo_diff,offense_spread,fg_pct_szn,fg_att_szn,off_games_prior,outcome
0,282410024058,282410024,28241002406,2008,1,Oregon State,Stanford,field goal,10,10,...,NaN,NaN,1612.0,1383.0,229.0,NaN,NaN,NaN,0.0,fg_made
1,282410066129,282410066,28241006616,2008,1,South Dakota State,Iowa State,field goal,4,19,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,fg_made
2,282410193115,282410193,28241019313,2008,1,Vanderbilt,Miami (OH),field goal,2,14,...,NaN,NaN,1473.0,1332.0,141.0,NaN,NaN,NaN,0.0,fg_made
3,282412117010,282412117,28241211701,2008,1,Central Michigan,Eastern Illinois,field goal,2,2,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,fg_made
4,282412117038,282412117,28241211704,2008,1,Eastern Illinois,Central Michigan,field goal,6,12,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,fg_made
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
43868,401869955142,401869955,4018699556,2026,2,Georgia State,Kennesaw State,field goal,9,9,...,0.592000,-0.392000,1241.0,1373.0,-132.0,7.5,1.0,1.0,1.0,fg_made
43869,401869961752,401869961,40186996138,2026,4,Old Dominion,James Madison,field goal,10,17,...,0.347333,-0.110333,1295.0,1777.0,-482.0,7.0,1.0,4.0,3.0,fg_made
43870,401871047658,401871047,40187104726,2026,3,Marshall,Missouri State,field goal,8,13,...,0.353000,0.124500,1388.0,1284.0,104.0,-4.0,0.0,1.0,2.0,fg_made
43871,40187104989,401871049,4018710495,2026,5,New Mexico State,Western Kentucky,field goal,15,34,...,0.442000,0.309250,1231.0,1242.0,-11.0,-1.5,0.5,6.0,4.0,fg_made


In [67]:
fg.info()

<class 'pandas.DataFrame'>
RangeIndex: 43873 entries, 0 to 43872
Data columns (total 30 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   play_id           43873 non-null  str    
 1   game_id           43873 non-null  int64  
 2   drive_id          43873 non-null  str    
 3   season            43873 non-null  int64  
 4   week              43873 non-null  int64  
 5   offense           43873 non-null  str    
 6   defense           43873 non-null  str    
 7   decision          43873 non-null  str    
 8   distance          43873 non-null  int64  
 9   yards_to_goal     43873 non-null  int64  
 10  kick_distance     43873 non-null  int64  
 11  is_goal_to_go     43873 non-null  bool   
 12  period            43873 non-null  int64  
 13  is_overtime       43873 non-null  bool   
 14  sec_left_in_half  43380 non-null  float64
 15  sec_left_in_game  43380 non-null  float64
 16  score_margin      43873 non-null  int64  
 17  is_h

In [68]:
# Create binary target variable
fg['fg_made'] = np.where(fg['outcome'] == 'fg_made', 1, 0)

# Drop identifer columns and group columns by data type for pipelines
df = fg.drop(columns=['play_id','game_id','drive_id','outcome','offense','defense','decision'],errors='ignore')
numerical_columns = df.select_dtypes(include='number').columns.to_list()
bool_columns = df.select_dtypes(include=['bool','object']).columns.to_list()

not_features = ['fg_made', 'season', 'week','kick_distance','yards_to_goal','elo_diff']
numerical_features = [c for c in numerical_columns if c not in not_features]

df[bool_columns] = df[bool_columns].astype('float')

# Split out target and features and create train test splits
X_all = df.drop(columns='fg_made')
y_all = df['fg_made']

training_season = range(2014,2024)
validate_season = 2024
test_season = 2025

train = df[df['season'].isin(training_season)]
valid = df[df['season'] == validate_season]
test = df[df['season'] == test_season]

# Valide the groups
print(train.shape)
print(valid.shape)
print(test.shape)

(23927, 24)
(2628, 24)
(2660, 24)


In [69]:
# Create Pipelines to prepare data for training
numerical_pipeline = make_pipeline(
    SimpleImputer(strategy='median'),
    StandardScaler(),
)

categorical_pipeline = make_pipeline(
    SimpleImputer(strategy='constant', fill_value=0)
)

preprocessor = ColumnTransformer([
    ('numerics', numerical_pipeline, numerical_features),
    ('binary',categorical_pipeline, bool_columns),
    ('distance',SplineTransformer(extrapolation='linear'), ['kick_distance'])
])

logistic_regression_pipeline = Pipeline([
    ('preprocess',preprocessor),
    ('model', LogisticRegression(
        max_iter=1000))]
)

# Prepare training data
feature_columns = numerical_features + bool_columns  + ['kick_distance']
X_train, y_train = train[feature_columns], train['fg_made']
X_valid, y_valid = valid[feature_columns], valid['fg_made']

# Fit and logistic regression model
logistic_regression_pipeline.fit(X_train, y_train)
valid_prob = logistic_regression_pipeline.predict_proba(X_valid)[:,1]
y_pred = logistic_regression_pipeline.predict(X_valid)

In [70]:
print(
    classification_report(
        y_valid,
        y_pred,
        labels=[0, 1],
        target_names=["Field Goal Missed", "Field Goal Made"],
        digits=4
    )
)

accuracy = accuracy_score(y_valid, y_pred)

# Metrics for the positive class: Default = 1
precision = precision_score(y_valid, y_pred, pos_label=1)
sensitivity = recall_score(y_valid, y_pred, pos_label=1)
f1 = f1_score(y_valid, y_pred, pos_label=1)
log_loss_score =  log_loss(y_valid, valid_prob)

auc = roc_auc_score(y_valid, valid_prob)

print(f"Accuracy       : {accuracy:.4f}")
print(f"Precision      : {precision:.4f}")
print(f"Default Recall : {sensitivity:.4f}")
print(f"F1 Score       : {f1:.4f}")
print(f"ROC AUC        : {auc:.4f}")
print(f"Log Loss       : {log_loss_score:4f}")
print("\n\n")

                   precision    recall  f1-score   support

Field Goal Missed     0.4868    0.2041    0.2876       632
  Field Goal Made     0.7871    0.9319    0.8534      1996

         accuracy                         0.7568      2628
        macro avg     0.6370    0.5680    0.5705      2628
     weighted avg     0.7149    0.7568    0.7173      2628

Accuracy       : 0.7568
Precision      : 0.7871
Default Recall : 0.9319
F1 Score       : 0.8534
ROC AUC        : 0.7054
Log Loss       : 0.506314





# Model vs Baseline
*How does the model performance compare to historical data*

In [75]:
# ---------- lookup table: historical make rate by distance band (training seasons only) ----------
band_edges = [0, 22, 27, 32, 37, 42, 47, 52, 57, 100]          # 5-yard bands; 57+ is open-ended (few long kicks)
band_labels = ['<22', '22-26', '27-31', '32-36', '37-41', '42-46', '47-51', '52-56', '57+']

def distance_band(kick_distance):
    return pd.cut(kick_distance, bins=band_edges, labels=band_labels, right=False)

# make rate per band, learned from the training seasons
lookup = (
    train.assign(band=distance_band(train['kick_distance']))
         .groupby('band', observed=True)['fg_made']
         .agg(make_rate='mean', kicks='size')
)
display(lookup.round(3))

# each validation kick gets its band's training rate (overall training rate if a band is missing)
lookup_prob = (
    distance_band(valid['kick_distance'])
        .map(lookup['make_rate'])
        .astype(float)
        .fillna(y_train.mean())
        .clip(0.01, 0.99)                                    # a 0% or 100% band would make log loss infinite
        .to_numpy()
)

# ---------- compare on the same validation kicks ----------
def evaluate(name, y_true, prob):
    return {
        'model': name,
        'log_loss': log_loss(y_true, prob),
        'brier': brier_score_loss(y_true, prob),
        'auc': roc_auc_score(y_true, prob),
        'mean_pred': prob.mean(),
        'actual_rate': y_true.mean(),
    }

naive_prob = np.full(len(y_valid), y_train.mean())

comparison = pd.DataFrame([
    evaluate('naive (training rate)', y_valid, naive_prob),
    evaluate('lookup table (5-yd bands)', y_valid, lookup_prob),
    evaluate('logistic + spline', y_valid, valid_prob),
]).set_index('model')
comparison.round(4)

,make_rate,kicks
band,,
<22,0.946,1559
22-26,0.912,3856
27-31,0.845,3815
32-36,0.776,3944
37-41,0.702,3959
42-46,0.608,3600
47-51,0.518,2413
52-56,0.426,713
57+,0.439,57


,log_loss,brier,auc,mean_pred,actual_rate
model,,,,,
naive (training rate),0.5522,0.1829,0.5000,0.7450,0.7595
lookup table (5-yd bands),0.5104,0.1689,0.6955,0.7261,0.7595
logistic + spline,0.5063,0.1672,0.7054,0.7268,0.7595
